In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/home/maria/delphi/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the data
df = pd.read_csv("../data/exposome.csv") 
df_dis = pd.read_csv("../data/labels_filtered.csv")

In [3]:
df.head()

,ukb_id,variable,answer_choice,third_person_narrative,question_answer
0,21001,bmi,Body Mass Index,Body Mass Index,Body Mass Index
1,1239,tobacco_smoker,"Yes, on most or all days",The person smokes on most or all days,Question: Do you smoke tobacco now? Answer: Ye...
2,1239,tobacco_smoker,Only occasionally,The person smokes only occasionaly,Question: Do you smoke tobacco now? Answer: On...
3,1239,tobacco_smoker,No,The person does not smoke,Question: Do you smoke tobacco now? Answer: No.
4,1558,alcohol,Daily or almost daily,The person drinks alcohol daily or almost daily,Question: About how often do you drink alcohol...


In [4]:
df_dis.head()

,code,description,meaning
0,NaN,Padding,Padding
1,NaN,No event,No event
2,NaN,Female,Female
3,NaN,Male,Male
4,NaN,BMI_low,BMI_low


We discretize the BMI variable below:

In [5]:
bmi_intervals = [
    "< 16.0",
    "16.0–17.0",
    "17.0–18.5",
    "18.5–25.0",
    "25.0–30.0",
    "30.0–35.0",
    "35.0–40.0",
    "≥ 40.0"
]

bmi_third_person = [f"The person has a Body Mass Index of {interval} kg/m^2" for interval in bmi_intervals]
bmi_question_answer = [f"Question: What is your Body Mass Index? Answer: {interval} kg/m^2." for interval in bmi_intervals]

In [6]:
bmi_third_person

['The person has a Body Mass Index of < 16.0 kg/m^2',
 'The person has a Body Mass Index of 16.0–17.0 kg/m^2',
 'The person has a Body Mass Index of 17.0–18.5 kg/m^2',
 'The person has a Body Mass Index of 18.5–25.0 kg/m^2',
 'The person has a Body Mass Index of 25.0–30.0 kg/m^2',
 'The person has a Body Mass Index of 30.0–35.0 kg/m^2',
 'The person has a Body Mass Index of 35.0–40.0 kg/m^2',
 'The person has a Body Mass Index of ≥ 40.0 kg/m^2']

In [7]:
bmi_question_answer

['Question: What is your Body Mass Index? Answer: < 16.0 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: 16.0–17.0 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: 17.0–18.5 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: 18.5–25.0 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: 25.0–30.0 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: 30.0–35.0 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: 35.0–40.0 kg/m^2.',
 'Question: What is your Body Mass Index? Answer: ≥ 40.0 kg/m^2.']

In [8]:
bmi_df = pd.DataFrame({
    "ukb_id": "21001",
    "variable": "bmi",
    "answer_choice": bmi_intervals,
    "third_person_narrative": bmi_third_person,
    "question_answer": bmi_question_answer
})

Replace BMI entry with discretized values:

In [9]:
df = df.drop(0, axis=0)
df = pd.concat([bmi_df, df])

To include death and padding tokens:

In [10]:
df = pd.concat([
    pd.DataFrame({
        "ukb_id": ["40000", pd.NA],
        "variable": ["death", "padding"],
        "answer_choice": ["death", "padding"],
        "third_person_narrative": ["The person has died", "No event"],
        "question_answer": ["The person has died", "No event"]
    }),
    df
])

In [11]:
df.head(15)

,ukb_id,variable,answer_choice,third_person_narrative,question_answer
0,40000,death,death,The person has died,The person has died
1,NaN,padding,padding,No event,No event
0,21001,bmi,< 16.0,The person has a Body Mass Index of < 16.0 kg/m^2,Question: What is your Body Mass Index? Answer...
1,21001,bmi,16.0–17.0,The person has a Body Mass Index of 16.0–17.0 ...,Question: What is your Body Mass Index? Answer...
2,21001,bmi,17.0–18.5,The person has a Body Mass Index of 17.0–18.5 ...,Question: What is your Body Mass Index? Answer...
3,21001,bmi,18.5–25.0,The person has a Body Mass Index of 18.5–25.0 ...,Question: What is your Body Mass Index? Answer...
4,21001,bmi,25.0–30.0,The person has a Body Mass Index of 25.0–30.0 ...,Question: What is your Body Mass Index? Answer...
5,21001,bmi,30.0–35.0,The person has a Body Mass Index of 30.0–35.0 ...,Question: What is your Body Mass Index? Answer...
6,21001,bmi,35.0–40.0,The person has a Body Mass Index of 35.0–40.0 ...,Question: What is your Body Mass Index? Answer...
7,21001,bmi,≥ 40.0,The person has a Body Mass Index of ≥ 40.0 kg/m^2,Question: What is your Body Mass Index? Answer...


We are going to compare 4 different scenarios for traning the model
1. We will use a general trained model for extracting the embeddings

    1.1. We will use the third person column for the exposome data

    1.2. We will use the question/answer column for the exposome data

2. We will use a medical trained model for extracting the embeddings 

    2.1. We will use the third person column for the exposome data

    2.2. We will use the question/answer column for the exposome data

# case 1 general model (obtained from huggingface, nº2 ranking date 16/03/2026)

In [12]:
model = SentenceTransformer("nvidia/llama-embed-nemotron-8b", device="cuda", trust_remote_code=True)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1351.69it/s]


 EXPOSOME EMBEDDINGS

In [13]:
emb_third = model.encode(
    df["third_person_narrative"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

emb_qa = model.encode(
    df["question_answer"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)


np.save("../data/emb_third_person.npy", emb_third)
np.save("../data/emb_qa.npy", emb_qa)
np.save("../data/ukb_id.npy", df["ukb_id"].values)

Batches: 100%|██████████| 3/3 [00:00<00:00,  7.09it/s]


In [14]:
# Thirds person 
np.load("../data/emb_third_person.npy")

array([[ 0.00379944,  0.00364685,  0.00842285, ..., -0.00101471,
         0.00741577, -0.00094223],
       [ 0.00656128, -0.00915527, -0.00964355, ...,  0.00982666,
         0.00643921, -0.00193787],
       [ 0.0072937 ,  0.0013504 ,  0.01428223, ...,  0.01123047,
         0.01721191,  0.00604248],
       ...,
       [ 0.01141357, -0.00753784, -0.00033569, ...,  0.0279541 ,
        -0.00130463,  0.00735474],
       [ 0.01251221, -0.00634766,  0.00218201, ...,  0.02539062,
        -0.00045967,  0.00537109],
       [ 0.0133667 , -0.00588989,  0.0020752 , ...,  0.02636719,
         0.00171661,  0.00613403]], dtype=float32)

In [15]:
len(emb_third)

73

In [16]:
# question answer
np.load("../data/emb_qa.npy")

array([[ 0.00390625,  0.00364685,  0.00842285, ..., -0.00125885,
         0.00735474, -0.00091934],
       [ 0.00643921, -0.00933838, -0.00982666, ...,  0.00976562,
         0.00653076, -0.00201416],
       [ 0.00017548, -0.01904297,  0.02368164, ...,  0.02587891,
         0.01806641,  0.0090332 ],
       ...,
       [ 0.00909424, -0.01989746,  0.00830078, ...,  0.046875  ,
         0.01855469,  0.0055542 ],
       [ 0.00982666, -0.01965332,  0.00958252, ...,  0.04541016,
         0.01953125,  0.00457764],
       [ 0.0100708 , -0.01965332,  0.00976562, ...,  0.04589844,
         0.0201416 ,  0.0050354 ]], dtype=float32)

In [17]:
len(emb_qa)

73

In [18]:
# ids
ids = np.load("../data/ukb_id.npy", allow_pickle=True)

DISEASES EMBEDDINGS

In [19]:
emb_diseases = model.encode(
    df_dis["meaning"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

np.save("../data/emb_diseases.npy", emb_diseases)
df_dis[["code"]].to_csv("../data/disease_codings.csv", index=False)

Batches: 100%|██████████| 40/40 [00:03<00:00, 13.26it/s]


In [20]:
np.load("../data/emb_diseases.npy")

array([[ 0.0255127 ,  0.02331543,  0.00692749, ..., -0.01550293,
         0.0090332 , -0.00405884],
       [ 0.00637817, -0.00946045, -0.00964355, ...,  0.00982666,
         0.00668335, -0.00205994],
       [ 0.01611328,  0.00680542,  0.00735474, ...,  0.00640869,
         0.0022583 , -0.00408936],
       ...,
       [ 0.03808594, -0.00405884,  0.00026131, ...,  0.01275635,
         0.00686646,  0.00805664],
       [ 0.0189209 ,  0.01312256, -0.00640869, ...,  0.00289917,
         0.01074219,  0.00723267],
       [ 0.00133514,  0.00260925,  0.00300598, ..., -0.01184082,
         0.0189209 , -0.00315857]], dtype=float32)

In [21]:
len(emb_diseases)

1270

In [22]:
np.load("../data/emb_diseases.npy").shape

(1270, 4096)

# case 2 medical model (obtained from MedTE a PubMed trained model)

In [23]:
model2 = SentenceTransformer("MohammadKhodadad/MedTE-cl15-step-8000")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4785.25it/s]


EXPOSOME EMBEDDINGS

In [24]:
emb_third_m2 = model2.encode(
    df["third_person_narrative"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

emb_qa_m2 = model2.encode(
    df["question_answer"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)


np.save("../data/emb_third_person_m2.npy", emb_third_m2)
np.save("../data/emb_qa_m2.npy", emb_qa_m2)

Batches: 100%|██████████| 3/3 [00:00<00:00, 41.58it/s]


In [25]:
# Thirds person 
np.load("../data/emb_third_person_m2.npy")

array([[ 0.01385665, -0.02147382,  0.01476166, ...,  0.05674188,
         0.01433197, -0.03461388],
       [-0.02192097, -0.05552665,  0.03227952, ...,  0.00379625,
        -0.00090103,  0.03411644],
       [-0.02010906, -0.0365889 ,  0.0203309 , ...,  0.01198648,
         0.00201555, -0.03103137],
       ...,
       [-0.06387946, -0.00151576,  0.00226893, ...,  0.02838259,
         0.06524542, -0.00794764],
       [-0.05918255, -0.00626234,  0.00137152, ...,  0.03051256,
         0.06997918, -0.00485148],
       [-0.05558091,  0.00529485,  0.0201252 , ...,  0.03009051,
         0.06701748,  0.00144941]], dtype=float32)

In [26]:
len(emb_third_m2)

73

In [27]:
# question answer
np.load("../data/emb_qa_m2.npy")

array([[ 0.01385665, -0.02147382,  0.01476166, ...,  0.05674188,
         0.01433197, -0.03461388],
       [-0.02192097, -0.05552665,  0.03227952, ...,  0.00379625,
        -0.00090103,  0.03411644],
       [ 0.00706832, -0.03416132,  0.01171785, ...,  0.05094352,
         0.04687867, -0.00032198],
       ...,
       [-0.02525862,  0.01218399,  0.0243988 , ...,  0.02559393,
         0.09050622, -0.02680081],
       [-0.02450451,  0.01233167,  0.02037469, ...,  0.03092748,
         0.09506334, -0.02526394],
       [-0.02572895,  0.01504942,  0.0236272 , ...,  0.02985463,
         0.09224647, -0.02497162]], dtype=float32)

In [28]:
len(emb_qa_m2)

73

DISEASE EMBEDDINGS

In [29]:
emb_diseases_m2 = model2.encode(
    df_dis["meaning"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

np.save("../data/emb_diseases_m2.npy", emb_diseases_m2)

Batches: 100%|██████████| 40/40 [00:00<00:00, 69.17it/s]


In [30]:
np.load("../data/emb_diseases_m2.npy")

array([[-0.00956056, -0.0083982 , -0.02410304, ..., -0.01617959,
        -0.04579102, -0.03346143],
       [-0.02192098, -0.05552668,  0.03227953, ...,  0.00379621,
        -0.000901  ,  0.03411645],
       [-0.04658561, -0.02359438, -0.04825744, ..., -0.01300313,
         0.05792036,  0.02693156],
       ...,
       [ 0.01898676, -0.01614128, -0.01828015, ..., -0.04992783,
         0.01965774,  0.01409905],
       [-0.05245044,  0.02560529, -0.04467658, ..., -0.09748561,
         0.06417527,  0.06027671],
       [ 0.05080681,  0.01470277, -0.01639245, ...,  0.04032998,
         0.01032149,  0.00566618]], dtype=float32)

In [31]:
len(emb_diseases_m2)

1270